# Transcrição com GPU no Google Colab

 Notebook para transcrever vídeos/áudios usando GPU gratuita do Colab.
 ~7x mais rápido que CPU para vídeos longos.

**Uso:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Instalar dependências
!pip install -q faster-whisper yt-dlp

In [ ]:
# 2. Carregar modelo (GPU)
from faster_whisper import WhisperModel

model = WhisperModel("base", device="cuda", compute_type="int8")
print("Modelo carregado na GPU!")

In [ ]:
# 3. Definir URL ou upload de arquivo
import os

# Opção A: URL do YouTube
VIDEO_URL = "https://www.youtube.com/watch?v=INSIRA_ID_AQUI"

# Opção B: Upload de arquivo local
# from google.colab import files
# uploaded = files.upload()  # selecionar arquivo de áudio/vídeo
# VIDEO_URL = list(uploaded.keys())[0]

In [ ]:
# 4. Download do áudio (se URL)
import subprocess

if VIDEO_URL.startswith("http"):
    !yt-dlp -x --audio-format mp3 -o "audio.%(ext)s" {VIDEO_URL}
    AUDIO_FILE = "audio.mp3"
else:
    AUDIO_FILE = VIDEO_URL

print(f"Áudio: {AUDIO_FILE}")

In [ ]:
# 5. Transcrever com GPU
import time

t0 = time.time()
segments, info = model.transcribe(AUDIO_FILE, batch_size=8, beam_size=5)

print(f"Idioma detectado: {info.language} ({info.language_probability:.2%})")
print(f"Duração: {info.duration:.1f}s")
print("Transcrevendo...")

# Consumir segmentos
segments_list = list(segments)
elapsed = time.time() - t0
print(f"Concluído em {elapsed:.1f}s ({info.duration/elapsed:.1f}x realtime)")

In [ ]:
# 6. Salvar SRT
def format_ts(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = round((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

srt_lines = []
for i, seg in enumerate(segments_list, 1):
    srt_lines.append(str(i))
    srt_lines.append(f"{format_ts(seg.start)} --> {format_ts(seg.end)}")
    srt_lines.append(seg.text.strip())
    srt_lines.append("")

srt_content = "\n".join(srt_lines)

with open("transcript.srt", "w", encoding="utf-8") as f:
    f.write(srt_content)

# Salvar texto corrido
plain_text = "\n".join(seg.text.strip() for seg in segments_list)
with open("full_text.txt", "w", encoding="utf-8") as f:
    f.write(plain_text)

print(f"Salvo: transcript.srt ({len(segments_list)} segmentos)")
print(f"Salvo: full_text.txt ({len(plain_text.split())} palavras)")

In [ ]:
# 7. Download dos arquivos
from google.colab import files

files.download("transcript.srt")
files.download("full_text.txt")